# Intro

Recoding of https://github.com/Daniel-EST/deep-steganography/ but in pytorch while adding below to fit paper being audited as code was not available to test

from paper: 500 training images, 50 validation images, and 50 test images, all uniformly resized to a manageable dimension of 64x64x3 pixels

text > Huffman > LSB > NN Embed > NN decode > LSB > Huffman > text

LSB - use pillow library (https://pypi.org/project/pillow/)

Huffman - use geeks for geeks

In [60]:
from LSBSteg.LSBSteg import LSBSteg
from dahuffman import HuffmanCodec

# code example from https://github.com/RobinDavid/LSB-Steganography libraru, lsb is using
# LSB 
def LSBEmbed(text,image):
    # Embed text into image using least sig bit method
    steg = LSBSteg(image)
    img_encoded = steg.encode_binary(text)
    return img_encoded

def LSBExtract(image):
    # pull least sig bit from image and reformat into embedded text
    steg = LSBSteg(image)
    extracted_text = steg.decode_binary()
    return extracted_text

# huffman serialization assisted with chatgpt for debugging, dahuffman use from PyPl example
def HuffmanEncode(data):
    # encode text into huffman tree + map
    codec = HuffmanCodec.from_data(data)
    encoded_data = codec.encode(data)

    # finally prepend all so tree:data is one block
    return codec, encoded_data

def HuffmanDecode(codec,data):
    # using Huffman map reconstruct text to original values and decode / unzip data
    decoded_data = codec.decode(data)
    return decoded_data

In [61]:
from datasets import load_dataset
import matplotlib.pyplot as plt
import os, certifi
import numpy as np
import cv2
import time
import random

os.environ['SSL_CERT_FILE'] = certifi.where()
def is_rgb(example):
    return example['image'].mode == 'RGB'
# option 1 uses "Hello World!" for all to reuse codec and make things simpler
temp_data = load_dataset("zh-plus/tiny-imagenet")
tiny_imageNet_rgb = temp_data.filter(is_rgb)

train_pool = tiny_imageNet_rgb['train']
test_pool = tiny_imageNet_rgb['valid']

ex_data = "Hello World!"
codec,encoded_data = HuffmanEncode(ex_data)

# 2. Sample from Training Pool
num_train = 500
train_indices = random.sample(range(len(train_pool)), num_train * 2)
c_train_imgs = [train_pool[i]['image'] for i in train_indices[:num_train]]
s_train_imgs = [train_pool[i]['image'] for i in train_indices[num_train:]]

# 3. Sample from Test Pool (Audit Set)
num_test = 50
test_indices = random.sample(range(len(test_pool)), num_test * 2)
c_test_imgs = [test_pool[i]['image'] for i in test_indices[:num_test]]
s_test_imgs = [test_pool[i]['image'] for i in test_indices[num_test:]]

s_train_steg = []
for img in s_train_imgs:
    np_img = np.array(img)
    s_train_steg.append( LSBEmbed(encoded_data,np_img))

s_test_steg = []
for img in s_test_imgs:
    np_img = np.array(img)
    s_test_steg.append( LSBEmbed(encoded_data,np_img))

print(f"Pools Ready: {len(c_train_imgs)} Train pairs, {len(c_test_imgs)} Test pairs.")



Pools Ready: 500 Train pairs, 50 Test pairs.


In [62]:
# steg NN
import torch
import torch.nn as nn
import torch.optim as optim

class PrepLayer(nn.Module):
    def __init__(self):
        super(PrepLayer, self).__init__()
        # Increases features from 3 to 150 as seen in your diagram
        self.conv = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(100, 150, kernel_size=3, padding=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.conv(x)

class HideLayer(nn.Module):
    def __init__(self):
        super(HideLayer, self).__init__()
        # Takes 150 (prep) + 3 (cover) = 153 channels
        self.conv = nn.Sequential(
            nn.Conv2d(153, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(100, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(50, 3, kernel_size=3, padding=1),
            nn.Sigmoid() # Keeps output in [0, 1] for image representation
        )

    def forward(self, secret_prepped, cover):
        # Concatenate along the channel dimension
        x = torch.cat((secret_prepped, cover), dim=1)
        return self.conv(x)

class RevealLayer(nn.Module):
    def __init__(self):
        super(RevealLayer, self).__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 50, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(50, 100, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(100, 3, kernel_size=3, padding=1),
            nn.Sigmoid()
        )

    def forward(self, container_image):
        return self.conv(container_image)

class SteganoModel(nn.Module):
    def __init__(self):
        super(SteganoModel, self).__init__()
        self.prep = PrepLayer()
        self.hide = HideLayer()
        self.reveal = RevealLayer()

    def forward(self, secret, cover):
        prepped_secret = self.prep(secret)
        container_image = self.hide(prepped_secret, cover)
        revealed_secret = self.reveal(container_image)
        return container_image, revealed_secret

# --- Initialization and Testing ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SteganoModel().to(device)

from torchvision import transforms

# Define the transform pipeline
steg_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((64, 64)),      
    transforms.ToTensor(),            
])

img_transform = transforms.Compose([
    transforms.Resize((64, 64)),      
    transforms.ToTensor(),            
])

In [65]:
# --- Training Configuration ---
num_epochs = 10
batch_size = 32
learning_rate = 0.001
beta = 1 # Weight for reconstruction (reveal) loss. Adjust between 0.5 and 1.0.
lambda_bit = 0.1

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Function to get a random batch from our sampled training lists
def get_training_batch(c_list, s_list, batch_size):
    indices = np.random.choice(len(c_list), batch_size)
    # Apply transform to each and stack into (B, 3, 64, 64)
    batch_c = torch.stack([img_transform(c_list[i]) for i in indices]).to(device)
    batch_s = torch.stack([steg_transform(s_list[i]) for i in indices]).to(device)
    return batch_c, batch_s

# --- Training Loop ---
model.train()
print(f"Starting Training on {len(c_train_imgs)} pairs...")

for epoch in range(num_epochs):
    epoch_loss = 0
    num_batches = len(c_train_imgs) // batch_size
    
    for i in range(num_batches):
        # 1. Prepare Data
        cover, secret = get_training_batch(c_train_imgs, s_train_steg, batch_size)
        
        # 2. Forward Pass
        optimizer.zero_grad()
        container, revealed = model(secret, cover)
        
        # 3. Calculate Loss
        # Loss 1: How much does the container look like the cover?
        loss_cover = criterion(container, cover)
        # Loss 2: How much does the revealed secret look like the original?
        loss_secret = criterion(revealed, secret)
        
        total_loss = loss_cover + (beta * loss_secret)

        # 4. Backward Pass
        total_loss.backward()
        optimizer.step()
        
        epoch_loss += total_loss.item()
        
    if (epoch + 1) % 5 == 0 or epoch == 0:
        avg_loss = epoch_loss / num_batches
        print(f"Epoch [{epoch+1}/{num_epochs}] | Total Loss: {avg_loss:.6f} | Cover MSE: {loss_cover.item():.6f} | Secret MSE: {loss_secret.item():.6f}")

print("Training Complete.")

Starting Training on 500 pairs...


KeyboardInterrupt: 

In [ ]:
#test
# Dummy inputs: (Batch, Channels, Height, Width)
i=0
for i in range(i,10):
    secret_input = steg_transform(s_test_steg[i]).unsqueeze(0).to(device)
    cover_input = img_transform(c_test_imgs[i]).unsqueeze(0).to(device)

    # 2. Model Inference
    model.eval()
    with torch.no_grad():
        container, revealed = model(secret_input, cover_input)

    # rev_img = extract_from_tensor(revealed)
    to_img = transforms.ToPILImage()
    rev_img = np.array(to_img(revealed.squeeze(0)))
    extracted_code = LSBExtract(rev_img)
    print(extracted_code)
    steg_text = HuffmanDecode(codec,extracted_code)

    print(f"steg text for image {i}: {steg_text}")

AttributeError: 'numpy.ndarray' object has no attribute 'channels'